In [1]:
# Imports
import pandas as pd
from random import randint
from src import *
from src.simulator import SIMULATOR

In [2]:
sim = SIMULATOR()

DEBUG = 1
# --------------------------------------------
#                DATA 
# --------------------------------------------
# DISCO-CGRA Configuration
nRCs = 4
nElementsPerVWRSlice = 32
nColsCGRA = 2

# Our test
ROWS_A = 32 # Multiple of 4 (M)
COLS_A = 32 # <= 32 (k)
COLS_B = 32 # Multiple of 4 (N)
# N*M min 128

ALPHA = 1
BETA = 0

# Blocks A:4x32 B:32x4
BLOCK_SIZE = 4

np.random.seed(0)   # opcional, para reproducibilidad

matrix_A = np.random.randint(0, 11, size=(ROWS_A * COLS_A))
matrix_B = np.random.randint(0, 11, size=(COLS_A * COLS_B))
matrix_C = np.random.randint(0, 11, size=(ROWS_A * COLS_B))
matrix_B_t = (matrix_B.reshape(COLS_A, COLS_B)).T.flatten()

# --------------------------------------------
#               KERNEL CONFIGURATION
# --------------------------------------------
kernel_path = './kernels/gemm_disco/v2/'
kernel_number = 1 
column_usage = [True, False] 
nInstrPerCol = 44
imem_add_start = 0 
srf_spm_addres = 0 
version=""

sim.kernel_config(column_usage, nInstrPerCol, imem_add_start, srf_spm_addres, kernel_number)

In [3]:
def printAsMatrix(array, rows, cols):
    for i in range(rows):
        print(array[i * cols:(i + 1) * cols])

In [ ]:
# --------------------------------------------
#              COMPILE ASM TO HEX
# --------------------------------------------
sim.compileAsmToHex(kernel_path, kernel_number, version=version)

# --------------------------------------------
#                 LOAD KERNEL
# --------------------------------------------

# This needs the hex instructions, if you don't provide them, generate then compiling the asm
sim.kernel_load(kernel_path, version=version + "_autogen", kernel_number=kernel_number)

Finally, we load the kernel into the internal memory of the specialized units and run it.

In [5]:
def gemm (in_A, in_B, nRowsA, nColsA, nColsB, alpha, beta):
    out = np.zeros(nRowsA*nColsB)
    for i in range(nRowsA):
        for j in range(nColsB):
            sum = 0
            for k in range(nColsA):
                sum += int(alpha * in_A[i*nColsA + k] * in_B[k*nColsB + j])
            out[i*nColsB + j] = sum + beta*out[i*nColsB + j]
    return [int(elem) for elem in out]

cpu_out = gemm(matrix_A, matrix_B, ROWS_A, COLS_A, COLS_B, ALPHA, BETA)

In [6]:
# Default SPM lines
srf_spm_line = 0

nLinesA = ROWS_A // BLOCK_SIZE
nLinesB = COLS_B // BLOCK_SIZE
nLinesC = nLinesA

c_spm_line_ini = 1 
a_spm_line_ini = c_spm_line_ini + nLinesC
b_spm_line_ini = a_spm_line_ini + nLinesA

In [7]:
def fillSPMWithSRF():
    # --------------------------------------------
    # SRF0 = SPMA
    # SRF1 = SPMC
    # SRF2 = M (= nBlocks A rows)
    # SRF3 = N (= nBlocks B cols)
    # SRF4 = k (= A cols = B rows) (=32 de momento para evitar padding)
    # SRF5 = alpha
    # SRF6 = beta
    # SRF7   Not used
    # --------------------------------------------

    # Default SRF values
    srf = [0 for i in range(N_ELEMS_PER_VWR)]

    # Col 0
    srf[0]  = a_spm_line_ini 
    srf[1]  = c_spm_line_ini 
    srf[2]  = ROWS_A // BLOCK_SIZE -1
    srf[3]  = COLS_B // BLOCK_SIZE -1
    srf[4]  = COLS_A -1
    srf[5]  = ALPHA
    srf[6]  = BETA
    srf[7]  = 0      # Not used
    # Col 1
    srf[8]  = 0      # Not used
    srf[9]  = 0      # Not used
    srf[10] = 0      # Not used
    srf[11] = 0      # Not used
    srf[12] = 0      # Not used
    srf[13] = 0      # Not used
    srf[14] = 0      # Not used
    srf[15] = 0      # Not used
    
    
    sim.setSPMLine(srf_spm_line, srf.copy())

In [8]:
# Weight stationary
# Fill entire matrixB
def fillSPMwithB_t(matrix_B_t, COLS_A, COLS_B): 
    b_spm_line = b_spm_line_ini

    for c in range(0, COLS_B//BLOCK_SIZE):
        sim.setSPMLine(b_spm_line, matrix_B_t[c*BLOCK_SIZE*COLS_A : (c+1)*BLOCK_SIZE*COLS_A].copy())
        b_spm_line += 1


In [9]:
def fillSPMWithA(matrix_A, ROWS_A, COLS_A): 
    a_spm_line = a_spm_line_ini

    for r in range(0, ROWS_A//BLOCK_SIZE):
        sim.setSPMLine(a_spm_line, matrix_A[r*BLOCK_SIZE*COLS_A:(r+1)*BLOCK_SIZE*COLS_A].copy())
        a_spm_line += 1



In [10]:
def fillSPMWithC(matrix_C, ROWS_A, COLS_B): 
    c_spm_line = c_spm_line_ini

    for r in range(0, ROWS_A//BLOCK_SIZE):
        sim.setSPMLine(c_spm_line, matrix_C[r*BLOCK_SIZE*COLS_B:(r+1)*BLOCK_SIZE*COLS_B].copy())
        c_spm_line += 1

In [11]:
def getCFromSPM():
    c_spm_line = c_spm_line_ini
    
    outs = []
    for i in range(ROWS_A // BLOCK_SIZE):
        outs.append(sim.getSPMLine(c_spm_line))
        c_spm_line += 1

    return outs

In [12]:
def runKernel(matrix_A, matrix_B_t, matrix_C, ROWS_A, COLS_A, COLS_B, max_iter=3000):
    # --------------------------------------------
    #               SIMULATE EXECUTION
    # --------------------------------------------
    show_lcu = []
    show_srf = []
    show_lsu = []
    show_rcs = [[],[],[],[]]
    show_mxcu = []
    display_ops = [show_lcu, show_lsu, show_mxcu, show_rcs, show_srf]

    # LOAD STATIC VALUES INTO SPM
    fillSPMwithB_t(matrix_B_t, COLS_A, COLS_B)
    fillSPMWithA(matrix_A, ROWS_A, COLS_A)
    fillSPMWithC(matrix_C, ROWS_A, COLS_B)
    fillSPMWithSRF()


    # BUFFERING AND COMPUTATION
    outs = []
    sim.kernel_load(kernel_path, version=version + "_autogen", kernel_number=kernel_number)

    if DEBUG:
        sim.displaySPMLine(0) # SRF
        sim.displaySPMLine(a_spm_line_ini) # A0
        sim.displaySPMLine(b_spm_line_ini) # B0
        sim.displaySPMLine(c_spm_line_ini) # C0

    # Launch kernel
    sim.run(kernel_number, display_ops=display_ops, max_iter=max_iter)

    # Get C from buffer
    outs.append(getCFromSPM())
    return outs


We can check it more rigorously. We can define our function in python and check that the output matches the CGRA output.

In [ ]:
# Run kernel
disco_out_aux = runKernel(matrix_A, matrix_B_t, matrix_C, ROWS_A, COLS_A, COLS_B, max_iter=300000)
disco_out = [int(x) for row in disco_out_aux for arr in row for x in arr]


In [ ]:
# Verify results
errors = 0
for i in range(len(cpu_out)):
    if cpu_out[i] != disco_out[i]:
        errors+=1
    
if errors == 0:
    print("The result is correct!")
else:
    print("Oops, something went wrong. There are " + str(errors) + " errors (out of " + str(len(cpu_out)) + " elements).")
    print("DISCO out:")
    printAsMatrix(disco_out, ROWS_A, COLS_B)
    print("Expected result:")
    printAsMatrix(cpu_out, ROWS_A, COLS_B)